# Prompt Engineering для анализа сентимента

Цель: изучить, как формулировки prompt и параметры генерации влияют на качество решения задачи классификации сентимента инструктивной LLM.

Задача — классификация сентимента русскоязычных текстов с использованием инструктивной языковой модели.

## 1. Скачиваем датасет

In [1]:
%pip install datasets transformers torch accelerate -q

from datasets import load_dataset

ds = load_dataset("sy-volkov/russian-sentiment-analysis-dataset")
print(ds)
print(ds["train"].features)
print(ds["train"][0])

Note: you may need to restart the kernel to use updated packages.


/Users/kornilovyv/Desktop/learning/otus/nlp_basic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['processed_text', 'Class', 'label', '__index_level_0__'],
        num_rows: 3873
    })
})
{'processed_text': Value('string'), 'Class': Value('string'), 'label': Value('int8'), '__index_level_0__': Value('int64')}
{'processed_text': 'Алексей, С Днем Рождения!!! Исполнения всего задуманного!!!', 'Class': 'G', 'label': 1, '__index_level_0__': 0}


## 2. Базовый prompt без system prompt

In [3]:
import torch

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Device:", DEVICE)

Device: mps


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).to(DEVICE)
model.eval()
print("Model loaded:", MODEL_NAME, "on", DEVICE)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 24013.78it/s]


Model loaded: Qwen/Qwen3-1.7B on mps


In [23]:
def make_prompt(text: str) -> str:
    return (
        f"Определи тональность следующего текста на русском языке. "
        f"Ответь одним словом: positive, negative или neutral.\n\n"
        f"Текст: {text}\n\n"
        f"Тональность:"
    )


def predict_greedy(text: str) -> str:
    """Жадная генерация (greedy decoding) без system prompt."""
    prompt = make_prompt(text)
    messages = [{"role": "user", "content": prompt}]

    tokenized = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,  # отключаем блок размышлений
        return_tensors="pt",
    )
    # apply_chat_template может вернуть BatchEncoding или тензор
    if hasattr(tokenized, 'input_ids'):
        input_ids = tokenized['input_ids'].to(model.device)
    else:
        input_ids = tokenized.to(model.device)

    prompt_length = input_ids.shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=20,
            do_sample=False,  # greedy decoding
        )

    generated = output_ids[0][prompt_length:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

In [24]:
# Быстрая проверка
test_text = "Это был отличный фильм, мне очень понравилось!"
test_output = repr(predict_greedy(test_text))
print(test_output)

'positive'


### Постобработка

In [27]:
import re

# N=0, G=1, B=2 — числовые лейблы датасета
LABEL_MAP = {
    "positive": 1, "позитивный": 1, "позитивно": 1, "положительный": 1,
    "negative": 2, "негативный": 2, "негативно": 2, "отрицательный": 2,
    "neutral": 0,  "нейтральный": 0, "нейтрально": 0,
    # output_numbered: числа совпадают с label датасета
    "0": 0, "1": 1, "2": 2,
}


def postprocess(raw_output: str) -> int | None:
    """Сопоставляет сырой вывод модели с числовым лейблом датасета (0/1/2)."""
    text = raw_output.strip()

    # JSON-формат: {"sentiment": "positive"}
    json_match = re.search(r'"sentiment"\s*:\s*"(\w+)"', text)
    if json_match:
        return LABEL_MAP.get(json_match.group(1).lower(), None)

    # убираем кавычки, берём первое слово
    cleaned = text.strip("\"'«»")
    first_word = re.split(r"[\s,.\n]+", cleaned)[0].lower()
    return LABEL_MAP.get(first_word, None)


In [30]:
# Проверка
examples = [
    ("positive", 1), ('"positive"', 1), ("негативно", 2),
    ("нейтральный текст", 0), ('{"sentiment": "negative"}', 2),
    ("1", 1), ("2", 2), ("хорошо", None),
]
for raw, expected in examples:
    result = postprocess(raw)
    status = "OK" if result == expected else "FAIL"
    print(f"{status}  {raw!r:35} -> {result}  (expected {expected})")

print("")
print(f"{test_output!r:30} -> {postprocess(test_output)}")

OK  'positive'                          -> 1  (expected 1)
OK  '"positive"'                        -> 1  (expected 1)
OK  'негативно'                         -> 2  (expected 2)
OK  'нейтральный текст'                 -> 0  (expected 0)
OK  '{"sentiment": "negative"}'         -> 2  (expected 2)
OK  '1'                                 -> 1  (expected 1)
OK  '2'                                 -> 2  (expected 2)
OK  'хорошо'                            -> None  (expected None)

"'positive'"                   -> 1


## 3. Вариации prompt

Сравниваем, как формулировка prompt влияет на качество классификации.
Каждый вариант запускается на одной и той же выборке из датасета.

In [31]:
# Словарь вариантов prompt: имя -> функция(text) -> str

PROMPT_VARIANTS = {

    # 1. Базовый
    "base": lambda text: (
        f"Определи тональность следующего текста на русском языке. "
        f"Ответь одним словом: positive, negative или neutral.\n\n"
        f"Текст: {text}\n\Тональность:"
    ),

    # 2. Менее строгий — не ограничиваем формат ответа
    "loose": lambda text: (
        f"Как ты оцениваешь тональность этого текста?\n\n"
        f"{text}"
    ),

    # 3. Более строгий — явный список, запрет пояснений
    "strict": lambda text: (
        f"Классифицируй тональность текста. "
        f"Выведи ТОЛЬКО одно слово из списка: positive, negative, neutral. "
        f"Никаких пояснений.\n\n"
        f"Текст: {text}\n\nОтвет:"
    ),

    # 4. Русские лейблы
    "russian_labels": lambda text: (
        f"Определи тональность текста. "
        f"Ответь одним словом: позитивный, негативный или нейтральный.\n\n"
        f"Текст: {text}\n\nТональность:"
    ),

    # 5. Инструкция на английском
    "english_instruction": lambda text: (
        f"Classify the sentiment of the following Russian text. "
        f"Answer with exactly one word: positive, negative, or neutral.\n\n"
        f"Text: {text}\n\nSentiment:"
    ),

    # 6. Структура: сначала текст, потом задача
    "context_first": lambda text: (
        f"Текст: {text}\n\n"
        f"Определи сентимент текста выше. "
        f"Ответь одним словом: positive, negative или neutral.\n\nСентимент:"
    ),

    # --- Форматы ввода ---

    # 7. Текст в кавычках
    "input_quoted": lambda text: (
        f"Определи сентимент текста. "
        f"Ответь одним словом: positive, negative или neutral.\n\n"
        f"Текст: \"{text}\"\n\nСентимент:"
    ),

    # 8. XML-подобные теги для обозначения текста
    "input_tagged": lambda text: (
        f"Определи сентимент текста внутри тегов <text>. "
        f"Ответь одним словом: positive, negative или neutral.\n\n"
        f"<text>{text}</text>\n\nСентимент:"
    ),

    # --- Форматы вывода ---

    # 9. JSON-формат вывода
    "output_json": lambda text: (
        f"Определи сентимент текста. "
        f"Ответь в формате JSON: {{\"sentiment\": \"positive\"|\"negative\"|\"neutral\"}}.\n\n"
        f"Текст: {text}\n\nОтвет:"
    ),

    # 10. Выбор из пронумерованного списка (числа совпадают с label датасета)
    "output_numbered": lambda text: (
        f"Определи сентимент текста. Выбери номер:\n"
        f"0 - neutral\n1 - positive\n2 - negative\n\n"
        f"Текст: {text}\n\nНомер:"
    ),
}


In [32]:
from sklearn.metrics import accuracy_score

def evaluate_prompt(prompt_fn, texts, labels):
    preds, none_count = [], 0
    for text in texts:
        messages = [{"role": "user", "content": prompt_fn(text)}]
        tokenized = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
        )
        input_ids = (
            tokenized["input_ids"] if hasattr(tokenized, "input_ids") else tokenized
        ).to(model.device)
        prompt_length = input_ids.shape[-1]

        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=20, do_sample=False)

        raw = tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True).strip()
        pred = postprocess(raw)
        if pred is None:
            none_count += 1
            pred = -1  # не засчитывается
        preds.append(pred)

    acc = accuracy_score(labels, preds)
    return acc, none_count



In [34]:
import random
import pandas as pd

random.seed(42)
SAMPLE_SIZE = 50
sample = random.sample(list(ds["train"]), SAMPLE_SIZE)
texts  = [ex["processed_text"] for ex in sample]
labels = [ex["label"] for ex in sample]

results = {}
for name, prompt_fn in PROMPT_VARIANTS.items():
    print(f"Evaluating: {name} ...")
    acc, none_count = evaluate_prompt(prompt_fn, texts, labels)
    results[name] = {"accuracy": acc, "unrecognized": none_count}
    print(f"  accuracy={acc:.3f}  unrecognized={none_count}/{SAMPLE_SIZE}")

df = pd.DataFrame(results).T
df["accuracy"] = df["accuracy"].map("{:.3f}".format)
df = df.sort_values("accuracy", ascending=False)
print(df.to_string())

Evaluating: base ...
  accuracy=0.820  unrecognized=0/50
Evaluating: loose ...
  accuracy=0.000  unrecognized=50/50
Evaluating: strict ...
  accuracy=0.820  unrecognized=0/50
Evaluating: russian_labels ...
  accuracy=0.640  unrecognized=0/50
Evaluating: english_instruction ...
  accuracy=0.940  unrecognized=0/50
Evaluating: context_first ...
  accuracy=0.800  unrecognized=0/50
Evaluating: input_quoted ...
  accuracy=0.860  unrecognized=0/50
Evaluating: input_tagged ...
  accuracy=0.920  unrecognized=0/50
Evaluating: output_json ...
  accuracy=0.880  unrecognized=0/50
Evaluating: output_numbered ...
  accuracy=0.340  unrecognized=33/50
                    accuracy  unrecognized
english_instruction    0.940           0.0
input_tagged           0.920           0.0
output_json            0.880           0.0
input_quoted           0.860           0.0
base                   0.820           0.0
strict                 0.820           0.0
context_first          0.800           0.0
russian_label

### Главные выводы 
Модель лучше понимает инструкции на английском; чёткое разграничение текста (теги, кавычки) улучшает результат; без ограничений на формат вывода (loose) postprocess полностью ломается.

## 4. Добавление system prompt

Варьируем system prompt в сочетании с разными user prompt и сравниваем с базовыми результатами.

In [35]:
SYSTEM_VARIANTS = {
    # 1. Короткий — только задача
    "sys_short": "Ты классификатор тональности текста.",

    # 2. С ролью эксперта
    "sys_expert": (
        "Ты эксперт по анализу тональности русскоязычных текстов. "
        "Твоя задача — определять тональность: positive, negative или neutral."
    ),

    # 3. Строгий — ограничение формата на уровне системы
    "sys_strict": (
        "Ты классификатор тональности. "
        "Отвечай ТОЛЬКО одним словом: positive, negative или neutral. "
        "Никаких пояснений, никакого другого текста."
    ),

    # 4. На английском
    "sys_english": (
        "You are a sentiment classifier for Russian texts. "
        "Classify each text as positive, negative, or neutral. "
        "Reply with exactly one word."
    ),
}


In [36]:
def evaluate_with_system(prompt_fn, system_prompt, texts, labels):
    preds, none_count = [], 0
    for text in texts:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": prompt_fn(text)},
        ]
        tokenized = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
        )
        input_ids = (
            tokenized["input_ids"] if hasattr(tokenized, "input_ids") else tokenized
        ).to(model.device)
        prompt_length = input_ids.shape[-1]

        with torch.no_grad():
            out = model.generate(input_ids, max_new_tokens=20, do_sample=False)

        raw = tokenizer.decode(out[0][prompt_length:], skip_special_tokens=True).strip()
        pred = postprocess(raw)
        if pred is None:
            none_count += 1
            pred = -1
        preds.append(pred)

    acc = accuracy_score(labels, preds)
    return acc, none_count


In [37]:
# Сравниваем все комбинации system prompt x user prompt
# Берём лучшие user prompts из предыдущего эксперимента
USER_PROMPTS_TO_TEST = {
    k: PROMPT_VARIANTS[k]
    for k in ["base", "strict", "english_instruction", "input_tagged"]
}

sys_results = {}
for sys_name, sys_prompt in SYSTEM_VARIANTS.items():
    for user_name, prompt_fn in USER_PROMPTS_TO_TEST.items():
        key = f"{sys_name} + {user_name}"
        print(f"Evaluating: {key} ...")
        acc, none_count = evaluate_with_system(prompt_fn, sys_prompt, texts, labels)
        sys_results[key] = {"accuracy": acc, "unrecognized": none_count}
        print(f"  accuracy={acc:.3f}  unrecognized={none_count}/{SAMPLE_SIZE}")

# Сравнение: без system prompt vs с system prompt
baseline = {k: results[k] for k in ["base", "strict", "english_instruction", "input_tagged"]}
baseline_df = pd.DataFrame(baseline).T
baseline_df.index = [f"[no system] {k}" for k in baseline_df.index]

sys_df = pd.DataFrame(sys_results).T

combined = pd.concat([baseline_df, sys_df])
combined["accuracy"] = combined["accuracy"].astype(float)
combined = combined.sort_values("accuracy", ascending=False)
combined["accuracy"] = combined["accuracy"].map("{:.3f}".format)
print(combined.to_string())


Evaluating: sys_short + base ...
  accuracy=0.960  unrecognized=0/50
Evaluating: sys_short + strict ...
  accuracy=0.860  unrecognized=0/50
Evaluating: sys_short + english_instruction ...
  accuracy=0.960  unrecognized=0/50
Evaluating: sys_short + input_tagged ...
  accuracy=0.460  unrecognized=26/50
Evaluating: sys_expert + base ...
  accuracy=0.760  unrecognized=1/50
Evaluating: sys_expert + strict ...
  accuracy=0.860  unrecognized=0/50
Evaluating: sys_expert + english_instruction ...
  accuracy=0.920  unrecognized=0/50
Evaluating: sys_expert + input_tagged ...
  accuracy=0.600  unrecognized=19/50
Evaluating: sys_strict + base ...
  accuracy=0.940  unrecognized=0/50
Evaluating: sys_strict + strict ...
  accuracy=0.960  unrecognized=0/50
Evaluating: sys_strict + english_instruction ...
  accuracy=0.900  unrecognized=0/50
Evaluating: sys_strict + input_tagged ...
  accuracy=0.980  unrecognized=0/50
Evaluating: sys_english + base ...
  accuracy=0.940  unrecognized=0/50
Evaluating: sys_

## Итоговые выводы

### Язык промпта
Промпт на английском языке стабильно даёт более высокую accuracy. Лучший одиночный результат — `english_instruction` (0.940), лучшая комбинация — `sys_english + strict` (0.980). Модель Qwen3 лучше следует инструкциям на английском, так как instruction-following часть обучена преимущественно на англоязычных данных.

### Влияние system prompt
Добавление system prompt в целом улучшает результат: большинство комбинаций превосходят baseline без system prompt (0.820). Наиболее эффективен строгий system prompt (`sys_strict`), явно ограничивающий формат вывода — он устраняет нераспознанные ответы даже там, где их было много.

### Формат ввода
Явное выделение текста улучшает результат: XML-теги (`input_tagged`, 0.920) и кавычки (`input_quoted`, 0.860) работают лучше базового варианта (0.820). Теги особенно хорошо сочетаются со строгим system prompt: `sys_strict + input_tagged` = 0.980.

### Формат вывода
Структурированный вывод (JSON, 0.880) работает хорошо при наличии соответствующего парсера в постобработке. Нумерованный список (`output_numbered`) и свободный ответ (`loose`) дают много нераспознанных ответов — без жёсткого ограничения формата модель не выдаёт предсказуемый вывод.

### Строгость ограничений
Явный запрет пояснений в user prompt (`strict`) сам по себе не даёт прироста над базовым вариантом (оба 0.820), но в сочетании с system prompt значительно усиливается: `sys_english + strict` = 0.980, `sys_strict + strict` = 0.960.


## 5. Лучшее из лучшего

Объединяем лучшие техники: английский язык, строгий system prompt, XML-теги в user prompt.

In [47]:
# sys_strict переведённый на английский
sys_strict_en = (
    "You are a sentiment classifier. "
    "Output ONLY one word: positive, negative, or neutral. "
    "No explanations, no other text."
)

# User prompts
strict_en        = lambda text: (
    f"Classify the sentiment. Output ONLY one word: positive, negative, or neutral.\n\n"
    f"Text: {text}\n\nSentiment:"
)
tagged_en        = lambda text: (
    f"Classify the sentiment of the text inside <text> tags. "
    f"Answer with exactly one word: positive, negative, or neutral.\n\n"
    f"<text>{text}</text>\n\nSentiment:"
)
tagged_strict_en = lambda text: (
    f"Classify the sentiment of the text inside <text> tags. "
    f"Output ONLY one word: positive, negative, or neutral. No explanations.\n\n"
    f"<text>{text}</text>\n\nSentiment:"
)

# Повторяем прошлые топ-комбинации для проверки воспроизводимости
experiments = {
    # Прошлые топы
    "sys_english   + strict (rerun)":        (SYSTEM_VARIANTS['sys_english'], PROMPT_VARIANTS['strict']),
    "sys_strict    + input_tagged (rerun)":  (SYSTEM_VARIANTS['sys_strict'],  PROMPT_VARIANTS['input_tagged']),
    # Новые комбинации
    "sys_strict_en + strict_en":             (sys_strict_en,                  strict_en),
    "sys_strict_en + tagged_en":             (sys_strict_en,                  tagged_en),
    "sys_english   + strict_en":             (SYSTEM_VARIANTS['sys_english'],  strict_en),
    "sys_english   + tagged_strict_en":      (SYSTEM_VARIANTS['sys_english'],  tagged_strict_en),
}


In [48]:
best_results = {}
for name, (sys_p, user_p) in experiments.items():
    print(f"Evaluating: {name} ...")
    acc, none_count = evaluate_with_system(user_p, sys_p, texts, labels)
    best_results[name] = {"accuracy": acc, "unrecognized": none_count}
    print(f"  accuracy={acc:.3f}  unrecognized={none_count}/{SAMPLE_SIZE}")

prev_best = {
    "sys_english + strict (prev)":       {"accuracy": 0.980, "unrecognized": 0},
    "sys_strict  + input_tagged (prev)": {"accuracy": 0.980, "unrecognized": 0},
}

combined = pd.DataFrame({**prev_best, **best_results}).T
combined["accuracy"] = combined["accuracy"].astype(float)
combined = combined.sort_values("accuracy", ascending=False)
combined["accuracy"] = combined["accuracy"].map("{:.3f}".format)
print(combined.to_string())


Evaluating: sys_english   + strict (rerun) ...
  accuracy=0.980  unrecognized=0/50
Evaluating: sys_strict    + input_tagged (rerun) ...
  accuracy=0.980  unrecognized=0/50
Evaluating: sys_strict_en + strict_en ...
  accuracy=0.860  unrecognized=0/50
Evaluating: sys_strict_en + tagged_en ...
  accuracy=0.720  unrecognized=0/50
Evaluating: sys_english   + strict_en ...
  accuracy=0.900  unrecognized=0/50
Evaluating: sys_english   + tagged_strict_en ...
  accuracy=0.880  unrecognized=0/50
                                     accuracy  unrecognized
sys_english + strict (prev)             0.980           0.0
sys_strict  + input_tagged (prev)       0.980           0.0
sys_english   + strict (rerun)          0.980           0.0
sys_strict    + input_tagged (rerun)    0.980           0.0
sys_english   + strict_en               0.900           0.0
sys_english   + tagged_strict_en        0.880           0.0
sys_strict_en + strict_en               0.860           0.0
sys_strict_en + tagged_en    

### Выводы

Оба прошлых топ-результата **воспроизвелись точно** — 0.980 в обоих случаях. Это не случайность.

**0.980 — стабильный результат** для двух комбинаций: `sys_english + strict` и `sys_strict + input_tagged`. Результат детерминированный (greedy decoding + фиксированная выборка).

**Ключевой паттерн — смешение языков**: английский system + русский user (или русский sys со структурой в user) работает лучше, чем однородные комбинации. Полностью английские варианты (`sys_strict_en + *`) стабильно хуже.

**Потолок на данной выборке — 0.980**. Дальнейшие эксперименты не дали прироста. Вероятно, оставшиеся 2% — объективно сложные примеры для модели такого размера.

**Итоговые лучшие комбинации:**
- `sys_english + strict` — английский строгий sys, русский строгий user без тегов
- `sys_strict + input_tagged` — русский строгий sys, русский user с XML-тегами
